In [ ]:
import pandas as pd
import numpy as np

# 1. CARREGAR PLANILHA
df = pd.read_csv('experimento_lambda_20260217_143915.csv')

print("Planilha carregada!")
print(f"Total de registros: {len(df)}")
print(f"Colunas: {df.columns.tolist()}\n")

# 2. DEFINIR VOCABULÁRIO DE FILMES
movie_vocabulary = {
    # Substantivos centrais
    'movie', 'film', 'cinema', 'flick', 'picture',

    # Elementos técnicos
    'director', 'acting', 'actor', 'actress', 'cast', 'performance',
    'cinematography', 'screenplay', 'script', 'plot', 'story',
    'scene', 'character', 'dialogue', 'narrative',

    # Gêneros e tipos
    'comedy', 'drama', 'horror', 'thriller', 'action',
    'documentary', 'animated', 'animation',

    # Avaliação técnica
    'directed', 'produced', 'starring', 'featuring',
    'pacing', 'editing', 'soundtrack', 'score',

    # Termos avaliativos (específicos de review)
    'masterpiece', 'brilliant', 'excellent', 'superb',
    'terrible', 'awful', 'disaster', 'boring',
    'entertaining', 'enjoyable', 'disappointing',
    'outstanding', 'mediocre', 'bland', 'cheesy',

    # Primeiro pessoa review
    'watched', 'viewing', 'recommend', 'rating',

    # Outros
    'oscar', 'award', 'sequel', 'franchise', 'remake',
    'blockbuster', 'indie', 'theatrical', 'premiere'
}

print(f"Vocabulário de domínio: {len(movie_vocabulary)} palavras\n")

# 3. FUNÇÃO PARA CALCULAR DOMAIN VOCAB SCORE
def calculate_domain_vocab_score(text):
    
    if pd.isna(text) or not text:
        return 0.0

    # Tokenizar em palavras (lowercase)
    words = text.lower().split()

    if len(words) == 0:
        return 0.0

    # Contar palavras do domínio
    domain_words = [w for w in words if w in movie_vocabulary]

    # Score: proporção de palavras do domínio
    score = len(domain_words) / len(words)

    return score

# 4. CALCULAR PARA TODA A PLANILHA
df['domain_vocab_score'] = df['texto_gerado'].apply(calculate_domain_vocab_score)

print("Domain Vocabulary Score calculado!\n")

# 5. AGREGAR POR λ
print("DOMAIN VOCABULARY SCORE POR λ")

domain_by_lambda = df.groupby('lambda')['domain_vocab_score'].agg(['mean', 'std', 'min', 'max'])

for lambda_val in [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]:
    row = domain_by_lambda.loc[lambda_val]
    mean_pct = row['mean'] * 100
    std_pct = row['std'] * 100

    # Barra visual
    barra = "█" * int(mean_pct / 2) + "░" * (50 - int(mean_pct / 2))

    print(f"λ={lambda_val:.1f}: {mean_pct:5.1f}% ± {std_pct:4.1f}% {barra[:30]}")

print("\n")

# 6. TOP PALAVRAS POR λ (BÔNUS!)
print("TOP 5 PALAVRAS MAIS FREQUENTES POR λ")

for lambda_val in [0.0, 1.0, 2.0, 3.0]:  # Amostrar alguns λ
    textos_lambda = df[df['lambda'] == lambda_val]['texto_gerado'].str.cat(sep=' ')
    words = textos_lambda.lower().split()

    # Apenas palavras do domínio
    domain_words_used = [w for w in words if w in movie_vocabulary]

    from collections import Counter
    top_words = Counter(domain_words_used).most_common(5)

    print(f"\nλ={lambda_val:.1f}:")
    for word, count in top_words:
        print(f"  - '{word}': {count}x")

In [ ]:

def calculate_repetition_rate(text, n=3):
   
    if pd.isna(text) or not text:
        return 0.0

    words = text.split()

    # Precisa de pelo menos n palavras
    if len(words) < n:
        return 0.0

    # Gerar n-grams
    ngrams = [tuple(words[i:i+n]) for i in range(len(words) - n + 1)]

    if len(ngrams) == 0:
        return 0.0

    # Calcular repetição
    unique_ngrams = len(set(ngrams))
    total_ngrams = len(ngrams)

    # Repetition rate: quanto MENOS único, mais repetição
    repetition = 1 - (unique_ngrams / total_ngrams)

    return repetition

# CALCULAR PARA TODA A PLANILHA
df['repetition_rate'] = df['texto_gerado'].apply(lambda x: calculate_repetition_rate(x, n=3))

# AGREGAR POR λ
repetition_by_lambda = df.groupby('lambda')['repetition_rate'].agg(['mean', 'std'])

for lambda_val in [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]:
    row = repetition_by_lambda.loc[lambda_val]
    mean_pct = row['mean'] * 100
    std_pct = row['std'] * 100

    # Barra visual (invertida - menos é melhor!)
    barra = "█" * int(mean_pct) + "░" * (50 - int(mean_pct))

    nivel = "Baixa" if mean_pct < 5 else "Média" if mean_pct < 15 else "Alta"

    print(f"λ={lambda_val:.1f}: {mean_pct:5.1f}% ± {std_pct:4.1f}% ({nivel}) {barra[:20]}")

print("\n" + "=" * 70)

# SALVAR PLANILHA ATUALIZADA
df.to_csv('experimento_lambda_com_metricas.csv', index=False)

In [ ]:
import re
import math
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ------------------------------------------------------------
CSV_PATH = "/content/experimento_yelp_20260331_002250.csv"

# Léxico de domínio Yelp
DOMAIN_VOCABULARY_YELP_RAW = {
    "food", "meal", "dish", "flavor", "taste", "portion", "menu",
    "appetizer", "entree", "dessert", "drink", "beverage", "sauce",
    "fresh", "stale", "spicy", "salty", "crispy", "soggy",
    "juicy", "tender", "greasy", "seasoned", "bland",
    "delicious", "tasty", "disgusting", "overcooked", "undercooked",
    "restaurant", "diner", "bistro", "eatery", "kitchen",
    "ambiance", "atmosphere", "waiter", "waitress", "chef",
    "reservation", "takeout", "delivery", "seating",
    "service", "staff", "attentive", "overpriced", "affordable",
    "ordered", "tasted", "served", "cooked", "visited",
    "recommend", "returned", "complained",
}

# Tokens gerados abaixo deste limiar serão classificados como EOS collapse.
EOS_COLLAPSE_THRESHOLD = 10

# ------------------------------------------------------------
# 2. FUNÇÕES DE NORMALIZAÇÃO
# ------------------------------------------------------------

def strip_accents(text: str) -> str:
    """Remove acentos de forma segura."""
    text = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in text if not unicodedata.combining(ch))


def normalize_basic(text: str) -> str:
    """
    Normalização básica:
    - lowercase
    - remove acentos
    - normaliza aspas
    - remove espaços duplicados
    """
    text = text.lower()
    text = strip_accents(text)
    text = text.replace("’", "'").replace("`", "'")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def conservative_lemma(token: str) -> str:
    """
    Lematização heurística CONSERVADORA.
    A ideia é capturar variações simples sem distorcer demais.

    Exemplos:
    - juicy! -> juicy   (pontuação removida antes)
    - recommended -> recommend
    - returned -> return
    - dishes -> dish
    - flavors -> flavor

    Não tenta resolver tudo; tenta ser confiável.
    """
    t = token

    # Não mexe em tokens muito curtos
    if len(t) <= 3:
        return t

    # Casos específicos frequentes no domínio
    irregular_map = {
        "recommended": "recommend",
        "recommends": "recommend",
        "recommending": "recommend",
        "returns": "return",
        "returning": "return",
        "returned": "return",
        "ordered": "ordered",
        "ordering": "order",
        "serves": "serve",
        "serving": "serve",
        "served": "served",
        "cooks": "cook",
        "cooking": "cook",
        "cooked": "cooked",
        "visited": "visited",
        "visiting": "visit",
        "complains": "complain",
        "complaining": "complain",
        "complained": "complained",
        "tastes": "taste",
        "tasting": "taste",
        "tasted": "tasted",
        "restaurants": "restaurant",
        "waiters": "waiter",
        "waitresses": "waitress",
        "chefs": "chef",
        "menus": "menu",
        "meals": "meal",
        "dishes": "dish",
        "flavors": "flavor",
        "portions": "portion",
        "desserts": "dessert",
        "drinks": "drink",
        "beverages": "beverage",
        "sauces": "sauce",
        "reservations": "reservation",
    }
    if t in irregular_map:
        return irregular_map[t]

    # Plural simples: foods -> food, flavors -> flavor
    if len(t) > 4 and t.endswith("s") and not t.endswith("ss"):
        singular = t[:-1]
        return singular

    # Passado em -ed muito simples
    if len(t) > 5 and t.endswith("ed"):
        base = t[:-2]
        # Ex.: recommended -> recommend
        if base.endswith("at") or base.endswith("iz") or base.endswith("end"):
            return base
        return t

    # Gerúndio em -ing muito simples
    if len(t) > 6 and t.endswith("ing"):
        base = t[:-3]
        return base

    return t


def tokenize_and_normalize(text: str) -> list[str]:
    
    text = normalize_basic(text)
    raw_tokens = re.findall(r"[a-zA-Z]+(?:'[a-zA-Z]+)?", text)
    normalized = [conservative_lemma(tok) for tok in raw_tokens]
    return normalized


def normalize_vocab(vocab: set[str]) -> set[str]:
    out = set()
    for item in vocab:
        toks = tokenize_and_normalize(item)
        # em geral cada item do léxico vira um token
        out.update(toks)
    return out


DOMAIN_VOCABULARY_YELP = normalize_vocab(DOMAIN_VOCABULARY_YELP_RAW)

# ------------------------------------------------------------
# 3. MÉTRICAS
# ------------------------------------------------------------

def compute_domain_metrics(text: str, vocab: set[str]) -> dict:
    
    if pd.isna(text):
        text = ""

    tokens = tokenize_and_normalize(text)
    total_words = len(tokens)

    hits_list = [tok for tok in tokens if tok in vocab]
    domain_hits = len(hits_list)

    domain_vocab_score = (domain_hits / total_words) if total_words > 0 else 0.0
    hits_per_50_tokens = (domain_hits / total_words * 50.0) if total_words > 0 else 0.0

    unique_hits = sorted(set(hits_list))

    return {
        "total_words_recomputed": total_words,
        "domain_hits": domain_hits,
        "domain_vocab_score_recomputed": domain_vocab_score,
        "domain_hits_per_50_tokens": hits_per_50_tokens,
        "domain_hit_types_count": len(unique_hits),
        "domain_hit_types": ", ".join(unique_hits),
    }


# ------------------------------------------------------------
# 4. LEITURA DO CSV
# ------------------------------------------------------------

csv_file = Path(CSV_PATH)
if not csv_file.exists():
    raise FileNotFoundError(f"CSV não encontrado em: {CSV_PATH}")

df = pd.read_csv(csv_file)

required_columns = ["lambda", "continuacao"]
missing = [c for c in required_columns if c not in df.columns]
if missing:
    raise ValueError(f"Colunas obrigatórias ausentes no CSV: {missing}")

# ------------------------------------------------------------
# 5. RECÁLCULO DAS MÉTRICAS
# ------------------------------------------------------------

metrics_df = df["continuacao"].apply(lambda t: compute_domain_metrics(t, DOMAIN_VOCABULARY_YELP))
metrics_df = pd.DataFrame(metrics_df.tolist())

df = pd.concat([df.reset_index(drop=True), metrics_df.reset_index(drop=True)], axis=1)

# Se o CSV já tiver num_tokens_gerados, vamos usar também.
# Caso não tenha, usamos total_words_recomputed como fallback.
if "num_tokens_gerados" not in df.columns:
    df["num_tokens_gerados"] = df["total_words_recomputed"]

# EOS collapse baseado em tokens gerados
df["eos_collapse_recomputed"] = df["num_tokens_gerados"] < EOS_COLLAPSE_THRESHOLD

# ------------------------------------------------------------
# 6. TABELAS-RESUMO
# ------------------------------------------------------------

summary = (
    df.groupby("lambda")
      .agg(
          n=("lambda", "size"),
          mean_domain_hits=("domain_hits", "mean"),
          std_domain_hits=("domain_hits", "std"),
          mean_domain_score=("domain_vocab_score_recomputed", "mean"),
          std_domain_score=("domain_vocab_score_recomputed", "std"),
          mean_hits_per_50=("domain_hits_per_50_tokens", "mean"),
          std_hits_per_50=("domain_hits_per_50_tokens", "std"),
          mean_total_words=("total_words_recomputed", "mean"),
          std_total_words=("total_words_recomputed", "std"),
          mean_num_tokens=("num_tokens_gerados", "mean"),
          std_num_tokens=("num_tokens_gerados", "std"),
          eos_collapse_rate=("eos_collapse_recomputed", "mean"),
      )
      .reset_index()
)

summary["eos_collapse_pct"] = summary["eos_collapse_rate"] * 100.0

# Formatação amigável
summary_rounded = summary.copy()
for col in summary_rounded.columns:
    if col not in ["lambda", "n"]:
        summary_rounded[col] = summary_rounded[col].astype(float).round(3)

print("=" * 90)
print("SUMMARY BY LAMBDA")
print("=" * 90)
print(summary_rounded.to_string(index=False))


# ------------------------------------------------------------
# 7. AMOSTRAS PARA INSPEÇÃO MANUAL
# ------------------------------------------------------------

print("MANUAL INSPECTION SAMPLES")

sample_cols = [
    "lambda", "seed", "prompt", "continuacao",
    "domain_hits", "domain_vocab_score_recomputed",
    "domain_hit_types"
]

available_sample_cols = [c for c in sample_cols if c in df.columns]

# 2 exemplos por lambda, priorizando hits altos
inspection_df = (
    df.sort_values(["lambda", "domain_hits"], ascending=[True, False])
      .groupby("lambda")
      .head(2)[available_sample_cols]
)

print(inspection_df.to_string(index=False, max_colwidth=120))

# ------------------------------------------------------------
# 8. EXPORTAR RESULTADOS
# ------------------------------------------------------------

output_dir = Path("/content/yelp_domain_analysis_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# CSV completo enriquecido
df.to_csv(output_dir / "yelp_enriched_generations.csv", index=False)

# resumo por lambda
summary_rounded.to_csv(output_dir / "yelp_summary_by_lambda.csv", index=False)

print("FILES EXPORTED")
print(f"Enriched generations: {output_dir / 'yelp_enriched_generations.csv'}")
print(f"Summary by lambda   : {output_dir / 'yelp_summary_by_lambda.csv'}")


SUMMARY BY LAMBDA
 lambda  n  mean_domain_hits  std_domain_hits  mean_domain_score  std_domain_score  mean_hits_per_50  std_hits_per_50  mean_total_words  std_total_words  mean_num_tokens  std_num_tokens  eos_collapse_rate  eos_collapse_pct
    0.0 60             0.950            1.702              0.023             0.042             1.148            2.079            41.433            3.638           50.000           0.000              0.000             0.000
    0.5 60             2.067            1.912              0.050             0.046             2.487            2.309            40.100            9.674           46.783          10.133              0.033             3.333
    1.0 60             2.800            1.783              0.079             0.054             3.974            2.696            35.667           12.629           42.200          13.917              0.067             6.667
    1.5 60             3.100            2.253              0.119             0.088        